In [0]:
DESCRIBE DETAIL samples.tpcds_sf1000.store_sales 

In [0]:
DESCRIBE DETAIL samples.tpcds_sf1000.date_dim

In [0]:
DESCRIBE DETAIL  samples.tpcds_sf1000.store

In [0]:
DESCRIBE DETAIL  samples.tpcds_sf1000.item

In [0]:
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1.store_sales ss
JOIN samples.tpcds_sf1.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1.item i
  ON ss.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
-- data is partitioned on ss_sold_date_sk, means daily 
CREATE or REPLACE TABLE gopal_db_workspace.default.daily_sales_agg
PARTITIONED BY (ss_sold_date_sk)
AS
SELECT
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk,
 SUM(ss_net_paid) sales
FROM samples.tpcds_sf1000.store_sales
GROUP BY 1,2,3;

In [0]:
 describe detail samples.tpcds_sf1000.store_sales

In [0]:
 describe detail gopal_db_workspace.default.daily_sales_agg

In [0]:
-- using pre-aggregated daily data
-- computed from daily aggregate
-- staging
CREATE or REPLACE TABLE gopal_db_workspace.default.yearly_sales_agg
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(a.sales) AS total_sales
FROM gopal_db_workspace.default.daily_sales_agg a
JOIN samples.tpcds_sf1000.date_dim d
  ON a.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON a.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON a.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
CREATE OR REPLACE TABLE gopal_db_workspace.default.monthly_sales_agg AS
SELECT
    d.d_year,
    d.d_moy,
    a.ss_store_sk,
    a.ss_item_sk,
    SUM(a.sales) AS monthly_sales
FROM gopal_db_workspace.default.daily_sales_agg a
JOIN samples.tpcds_sf1000.date_dim d
  ON a.ss_sold_date_sk = d.d_date_sk
GROUP BY
    d.d_year,
    d.d_moy,
    a.ss_store_sk,
    a.ss_item_sk;

In [0]:
describe detail gopal_db_workspace.default.monthly_sales_agg

In [0]:
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON ss.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
SELECT  /*+ BROADCAST(d,s,i) */
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON ss.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
WITH sales_agg AS (
SELECT
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk,
 SUM(ss_net_paid) sales
FROM samples.tpcds_sf1000.store_sales
GROUP BY
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk
)
SELECT
 d.d_year,
 s.s_state,
 i.i_category,
 SUM(a.sales)
FROM sales_agg a
JOIN samples.tpcds_sf1000.date_dim d
 ON a.ss_sold_date_sk=d.d_date_sk
JOIN samples.tpcds_sf1000.store s
 ON a.ss_store_sk=s.s_store_sk
JOIN samples.tpcds_sf1000.item i
 ON a.ss_item_sk=i.i_item_sk
GROUP BY 1,2,3;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
EXPLAIN SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1.store_sales ss
JOIN samples.tpcds_sf1.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1.item i
  ON ss.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
-- if any duplicates in combinatioin

SELECT
 d.d_year,
 s.s_state,
 i.i_category,
 COUNT(*) cnt
FROM samples.tpcds_sf1.store_sales ss
JOIN samples.tpcds_sf1.date_dim d
 ON ss.ss_sold_date_sk=d.d_date_sk
JOIN samples.tpcds_sf1.store s
 ON ss.ss_store_sk=s.s_store_sk
JOIN samples.tpcds_sf1.item i
 ON ss.ss_item_sk=i.i_item_sk
GROUP BY 1,2,3
ORDER BY cnt DESC;

In [0]:
%python
spark.conf.get("spark.sql.files.maxPartitionBytes")

In [0]:
SET spark.sql.files.maxPartitionBytes=270532608;

In [0]:
SET spark.sql.shuffle.partitions=200;

In [0]:
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(a.sales) AS total_sales
FROM gopal_db_workspace.default.sales_agg a
JOIN samples.tpcds_sf1000.date_dim d
  ON a.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON a.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON a.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category
ORDER BY total_sales DESC;

In [0]:
-- staging: pre-aggregate at query grain
CREATE OR REPLACE TABLE sales_cube AS
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
 ON ss.ss_sold_date_sk=d.d_date_sk
JOIN samples.tpcds_sf1000.store s
 ON ss.ss_store_sk=s.s_store_sk
JOIN samples.tpcds_sf1000.item i
 ON ss.ss_item_sk=i.i_item_sk
GROUP BY 1,2,3;

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:715)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:435)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:435)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
CREATE  OR REPLACE TABLE gopal_db_workspace.default.sales_agg as 
SELECT
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk,
 SUM(ss_net_paid) sales
FROM samples.tpcds_sf1000.store_sales
GROUP BY
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk


In [0]:
CREATE OR REPLACE TABLE sales_agg2_category AS
SELECT
  d.d_year,
  s.s_state,
  i.i_category,
  SUM(ss.ss_net_paid) total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
 ON ss.ss_sold_date_sk=d.d_date_sk
JOIN samples.tpcds_sf1000.store s
 ON ss.ss_store_sk=s.s_store_sk
JOIN samples.tpcds_sf1000.item i
 ON ss.ss_item_sk=i.i_item_sk
GROUP BY 1,2,3;

In [0]:
describe detail gopal_db_workspace.default.sales_agg 

In [0]:
-- spark.sql.files.maxRecordsPerFile
OPTIMIZE gopal_db_workspace.default.sales_agg ;

In [0]:
describe detail gopal_db_workspace.default.sales_agg 

In [0]:
DESCRIBE DETAIL samples.tpcds_sf1000.store_sales;

In [0]:
SELECT ss_sold_date_sk from samples.tpcds_sf1000.store_sales LIMIT 5

In [0]:
SELECT *
FROM samples.tpcds_sf1000.date_dim
WHERE d_date_sk = 2450816;

In [0]:
select count(*) from samples.tpcds_sf1000.date_dim

In [0]:
SHOW PARTITIONS samples.tpcds_sf1000.store_sales;

In [0]:
SELECT distinct (d_year)
FROM samples.tpcds_sf1000.date_dim
LIMIT 200;

In [0]:
SELECT *
FROM samples.tpcds_sf1000.date_dim
LIMIT 20;

In [0]:
EXPLAIN FORMATTED
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON ss.ss_item_sk = i.i_item_sk
WHERE d.d_year BETWEEN 1998 AND 2002
GROUP BY
 d.d_year,
 s.s_state,
 i.i_category
ORDER BY total_sales DESC;

In [0]:
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(ss.ss_net_paid) AS total_sales
FROM samples.tpcds_sf1000.store_sales ss
JOIN samples.tpcds_sf1000.date_dim d
  ON ss.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON ss.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON ss.ss_item_sk = i.i_item_sk
WHERE d.d_year BETWEEN 1998 AND 2002
GROUP BY
 d.d_year,
 s.s_state,
 i.i_category


In [0]:
CREATE OR REPLACE TABLE gopal_db_workspace.default.yearly_sales_from_daily AS
SELECT
    d.d_year,
    s.s_state,
    i.i_category,
    SUM(a.sales) AS total_sales
FROM gopal_db_workspace.default.daily_sales_agg a
JOIN samples.tpcds_sf1000.date_dim d
  ON a.ss_sold_date_sk = d.d_date_sk
JOIN samples.tpcds_sf1000.store s
  ON a.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON a.ss_item_sk = i.i_item_sk
GROUP BY
    d.d_year,
    s.s_state,
    i.i_category;

In [0]:
MERGE INTO gopal_db_workspace.default.daily_sales_agg t
USING (

SELECT
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk,
 SUM(ss_net_paid) sales
FROM samples.tpcds_sf1000.store_sales
WHERE ss_sold_date_sk = (
   SELECT d_date_sk
   FROM samples.tpcds_sf1000.date_dim
   WHERE d_date = DATE '2000-01-16'
)
GROUP BY
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk

) s
ON t.ss_sold_date_sk = s.ss_sold_date_sk
AND t.ss_store_sk = s.ss_store_sk
AND t.ss_item_sk = s.ss_item_sk

WHEN MATCHED THEN
UPDATE SET
 t.sales = s.sales

WHEN NOT MATCHED THEN
INSERT (
 ss_sold_date_sk,
 ss_store_sk,
 ss_item_sk,
 sales
)
VALUES (
 s.ss_sold_date_sk,
 s.ss_store_sk,
 s.ss_item_sk,
 s.sales
);

In [0]:
  SELECT d_date_sk
   FROM samples.tpcds_sf1000.date_dim
   WHERE d_date = DATE '2000-01-16'

In [0]:
SELECT count(*)
FROM samples.tpcds_sf1000.store_sales
WHERE ss_sold_date_sk = 2451560;

In [0]:
CREATE OR REPLACE TABLE gopal_db_workspace.default.yearly_sales_from_monthly AS
SELECT
    m.d_year,
    s.s_state,
    i.i_category,
    SUM(m.monthly_sales) AS total_sales
FROM gopal_db_workspace.default.monthly_sales_agg m
JOIN samples.tpcds_sf1000.store s
  ON m.ss_store_sk = s.s_store_sk
JOIN samples.tpcds_sf1000.item i
  ON m.ss_item_sk = i.i_item_sk
GROUP BY
    m.d_year,
    s.s_state,
    i.i_category;